## Bellman Equation and MPD Value & Policy Iteration for Funded Account Modelling 

### 1) Introduction

$$V(s) = \max_a \mathbb{E}[R + \gamma V(s')]$$

We will be using a Markov Decision Process (MDP) as the core mathematical framework for model sequential decision-making in this project. To better understand how formal MDP is applied in the prop firm setting, we will frist begin by using a toy scenario:

A token on a ladder of dollar states {0, 1, 2, 3, 4}. States 0 and 4 are absorbing (game over --> win or lose). From any interior state we pick one of two actions each round:
- Tight: +1 with prob 0.7, -1 with prob 0.3
- Wide: +2 with prob 0.4, -1 with prob 0.6

A win is achieved by reaching 4; and a loss is at 0. (This is essentially the funded account in miniature: 0 = breach, 4 = payout, Tight/Wide = low/high RR)
We now formalize this toy problem as an MDP:
A MDP is formally a five-tuple $$(S, A, P, R, \gamma)$$
- $S$: The set of states
- $A$: The set of actions
- ${P(s' | s, a)}$: Transition kernel: Probability of landing in $s$ given you are in $s$ and take $a$
- $R(s,a)$: Reward
- $ \gamma$: Discount factor

### 2) Step 1 - Toy Example Initialization

The five-tuple in the toy example is defined exactly as:
- $S$: {0, 1, 2, 3, 4}
- $A$: {Tight, Wide}
- ${P(s' | s, a)}$: Transition kernel: Probability of landing in $s$ given you are in $s$ and take $a$
- $R(s,a)$: 1 for reaching 4
- $ \gamma$: 1, we don't discount the toy scenario

$$P(s' \mid s=2,\ \text{Tight}) = \begin{cases} 0.7 & s'=3 \\ 0.3 & s'=1 \end{cases}$$

$$P(s' \mid s=2,\ \text{Wide}) = \begin{cases} 0.4 & s'=4 \\ 0.6 & s'=1 \end{cases}$$

### 3) Step 2 - The MDP tuple: Physics vs. Objective

| MDP object | Category | Toy ladder | When Relating to Prop firm |
|---|---|---|---|
| $\mathcal{S}$ (states) | Physics (the world) | position on $\{0,1,2,3,4\}$ | account state $(x, m, \ell, k)$: equity, running peak, lock flag, withdrawals taken |
| $\mathcal{A}$ (actions) | Physics (the world) | $\{\text{Tight}, \text{Wide}\}$ | $(r, \rho, w)$: risk size, reward ratio (RR), withdraw-or-push |
| $P(s' \mid s,a)$ (transitions) | Physics (the world) | Tight/Wide kernels ($0.7/0.3$, $0.4/0.6$) | empirical, cost-adjusted return distribution $\mathcal{D}_\rho$: where a trade takes you, and where the edge lives |
| $R(s,a)$ (reward) | Objective (what we care about) | $1$ for reaching $4$ | cash withdrawn (extraction), or $1$ for recouping the budget (survival) |
| $\gamma$ (discount) | Objective (what we care about) | $1$ (no time-preference) | $\beta < 1$: time-value of money plus hazard of losing the account |

Note: The "floor" of a funded account does not get its own MDP object, it is part of $S$, through the running peak $m$, and enforces itself as an absorbing boundary condition on $P$ (breach = the killed state). The barrier isn't an additional constraint, it is a property of the state space and transitions. 

### 4) Step 3 - Markov Property

A process $\{s_t\}$ is Markov if for all $t$,

$$\Pr\big(s_{t+1} \mid s_t, s_{t-1}, \dots, s_0, a_t\big) = \Pr\big(s_{t+1} \mid s_t, a_t\big).$$

Conditioning on the full history gives the same distribution as conditioning on the current state alone. Once you know $s_t$, earlier states add **nothing** about $s_{t+1}$. This is what lets us write the kernel as $P(s' \mid s, a)$ instead of $P(s' \mid s_0, \dots, s_t, a)$. **The next state depends only on the current state, not on the path that led there**.


Whether the property holds depends on what goes in the state. A variable belongs in the state when $P$ or $R$ depends on it and the other state variables don't already capture it.

With equity $x$ alone, it fails. The floor is $F = m - D$, which depends on the peak $m = \max_{\tau \le t} x_\tau$, a function of the whole path. Two histories can reach the same $x$ with different $m$:

$$x_t = 2000,\ m_t = 2000 \Rightarrow F = 0 \qquad \text{vs.} \qquad x_t = 2000,\ m_t = 4000 \Rightarrow F = 2000.$$

Same equity, different breach distance, different next-step probabilities. So $\Pr(s_{t+1} \mid x_t, a_t)$ is not well-defined.

The fix is to put the path-dependent quantities into the state:

$$s_t = (x_t, m_t, \ell_t, k_t).$$

$m_t$ makes the floor a function of the state, $k_t$ makes the payout cap and forced conversion a function of the state, and $\ell_t$ carries lock status. Everything the next transition needs is now computable from $s_t$, so the process is Markov.

Test: given only $s$ with all history hidden, are the next-step probabilities determined? No with equity alone (floor unknown). Yes with $(x, m, \ell, k)$.

### 5) Grading a Fixed Policy

$$V^\pi(s) = \mathbb{E}\big[\, R_0 + \gamma R_1 + \gamma^2 R_2 + \cdots \,\big]$$

where $R_0 = R(s, \pi(s))$ is the reward on this step and $R_1, R_2, ...$ are the rewards on future steps, all under $\pi$

Derivations:

$$(\gamma R_1 + \gamma^2 R_2 + + \gamma^3 R_3 + \cdots)  = \gamma (R_1 + \gamma R_2 + \gamma R_1 + \cdots)$$

We can see the $R_0$ term drops off and now $V^\pi$ is evaluated at the next state, the one in which we land right after taking a step from $s$. When $R_1$ is the next reward to be collected, we are standing at $s'$, and the discounted sum from there $R_1 + \gamma R_2 + \cdots$ is the value of $s'$. Hence the tail is $\gamma V^\pi(s')$

From $s$ under action $\pi(s)$, we don't land in a fixed $s'$, we land in a random one, distributed by the kernel $P(s' | s, \pi(s))$, hence the tail is for the average of $\gamma V^\pi (s')$ over where we might land.

Now, the discounted averaged value of the next state is estimated by:

$$V^\pi(s) = R(s,\pi(s)) + \gamma \sum_{s'} P\big(s'\mid s,\pi(s)\big)\, V^\pi(s')$$

Which is the **Bellman expectation equation**




### 6) Grading a Fixed Policy - Toy Example

Fix the policy Tight at every state. Tight's kernel: $+1$ with probability $0.7$, $-1$ with probability $0.3$. Boundaries $V(0) = 0$, $V(4) = 1$.

The Bellman expectation equation at each interior state:

$$V^\pi(1) = 0.7\,V^\pi(2) + 0.3\,V^\pi(0) = 0.7\,V^\pi(2)$$
$$V^\pi(2) = 0.7\,V^\pi(3) + 0.3\,V^\pi(1)$$
$$V^\pi(3) = 0.7\,V^\pi(4) + 0.3\,V^\pi(2) = 0.7 + 0.3\,V^\pi(2)$$

Substitute $V^\pi(1)$ and $V^\pi(3)$ into the middle equation:

$$V^\pi(2) = 0.7\,(0.7 + 0.3\,V^\pi(2)) + 0.3\,(0.7\,V^\pi(2))$$
$$V^\pi(2) = 0.49 + 0.21\,V^\pi(2) + 0.21\,V^\pi(2)$$
$$0.58\,V^\pi(2) = 0.49$$
$$V^\pi(2) = 0.845$$

Back-substitute:

$$V^\pi(1) = 0.592, \qquad V^\pi(2) = 0.845, \qquad V^\pi(3) = 0.954$$